# MAKE A COPY DULU YA GUYS 📌

Sebelum mulai, buka notebook ini di **Google Colab** lalu lakukan:

**File → Save a copy in Drive**

Kenapa harus di-*copy*?
- supaya versi asli tetap aman,
- supaya kamu bisa edit bagian `TODO`,
- supaya hasil eksperimenmu tersimpan di Drive sendiri.

# Challenge Notebook — CIFAR-10: MLP vs CNN with TensorFlow

Notebook ini adalah **challenge lanjutan** untuk peserta workshop **Intro to Deep Learning & Computer Vision**.

Di challenge ini kamu **tidak perlu mulai dari nol**.  
Tugasmu adalah **mengisi bagian-bagian yang masih kosong (`TODO`)**, lalu menjalankan eksperimen sampai selesai.

---

## Target challenge

Pada akhir notebook ini, kamu diharapkan bisa:

1. memuat dan memahami dataset **CIFAR-10**,
2. melakukan **preprocessing** data gambar,
3. melatih **model MLP** sebagai baseline,
4. melatih **model CNN** untuk data citra,
5. membandingkan performa **MLP vs CNN**,
6. melakukan **inferensi** menggunakan model terbaik,
7. membuat kesimpulan sederhana dari hasil eksperimen.

---

## Output yang diharapkan

| Bagian | Yang harus ada |
|---|---|
| Eksplorasi data | Visualisasi sampel gambar CIFAR-10 |
| Preprocessing | Data ternormalisasi dan siap untuk MLP & CNN |
| Model 1 | MLP terlatih |
| Model 2 | CNN terlatih |
| Evaluasi | Accuracy / loss / confusion matrix / sample predictions |
| Perbandingan | Kesimpulan model mana yang lebih baik dan kenapa |
| Inferensi | Prediksi pada beberapa gambar test |

> **Catatan:** Fokus challenge ini bukan bikin semuanya dari nol, tapi memahami **alur kerja computer vision end-to-end** dan melihat perbedaan representasi **MLP vs CNN**.

## Aturan pengerjaan

- Kerjakan notebook **dari atas ke bawah**.
- Isi bagian yang diberi tanda **`TODO`**.
- Jangan hapus helper function yang sudah disediakan.
- Kalau ada error, baca pesan error-nya baik-baik — biasanya ada petunjuk bagian mana yang belum diisi.
- Setelah selesai, tulis kesimpulanmu di bagian refleksi akhir.

---

## Strategi pengerjaan yang disarankan

1. Pastikan runtime Colab sudah memakai **GPU** bila tersedia.
2. Jalankan bagian **setup** dan **helper** terlebih dahulu.
3. Isi TODO untuk **load data** dan **preprocessing**.
4. Latih **MLP** dulu sebagai baseline.
5. Latih **CNN**, lalu bandingkan hasilnya.
6. Jalankan bagian **visualisasi** dan **inferensi**.
7. Kerjakan **bonus** kalau masih ada waktu.

In [ ]:

# ============================================
# 1. Setup Environment & Import Libraries
# ============================================

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("TensorFlow version:", tf.__version__)
print("GPU detected:", tf.config.list_physical_devices('GPU'))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Konfigurasi eksperimen

Untuk mempercepat runtime saat challenge, kamu bisa mulai dari **subset data** terlebih dahulu.  
Kalau semua sudah berjalan baik, kamu bisa coba ulang dengan **full dataset**.

### Saran
- `USE_SUBSET = True` saat debugging awal
- `USE_SUBSET = False` saat final run

In [ ]:

# ============================================
# 2. Konfigurasi
# ============================================

USE_SUBSET = True

TRAIN_SAMPLES = 20000   # dipakai kalau USE_SUBSET=True
TEST_SAMPLES  = 4000    # dipakai kalau USE_SUBSET=True

BATCH_SIZE = 64
EPOCHS_MLP = 10
EPOCHS_CNN = 12
VAL_RATIO = 0.1

IMG_HEIGHT = 32
IMG_WIDTH = 32
IMG_CHANNELS = 3
NUM_CLASSES = 10

## Helper functions

Bagian ini sudah disediakan agar kamu tidak perlu menulis ulang fungsi visualisasi dari nol.
Jalankan saja cell di bawah ini.

In [ ]:

# ============================================
# 3. Helper Functions
# ============================================

CLASS_NAMES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

def show_class_distribution(labels, title="Distribusi Label"):
    labels = np.array(labels).reshape(-1)
    counts = pd.Series(labels).value_counts().sort_index()
    df_counts = pd.DataFrame({
        "class_id": counts.index,
        "class_name": [CLASS_NAMES[i] for i in counts.index],
        "count": counts.values
    })
    display(df_counts)

    plt.figure(figsize=(10, 4))
    plt.bar(df_counts["class_name"], df_counts["count"])
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.show()

def plot_history(history, title_prefix="Model"):
    hist = history.history
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(hist["accuracy"], label="train")
    plt.plot(hist["val_accuracy"], label="val")
    plt.title(f"{title_prefix} Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(hist["loss"], label="train")
    plt.plot(hist["val_loss"], label="val")
    plt.title(f"{title_prefix} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()

def plot_conf_matrix(y_true, y_pred, class_names, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

def show_prediction_grid(images, true_labels, pred_probs, class_names, n=12):
    true_labels = np.array(true_labels).reshape(-1)
    pred_labels = np.argmax(pred_probs, axis=1)

    n = min(n, len(images))
    cols = 4
    rows = int(np.ceil(n / cols))

    plt.figure(figsize=(4 * cols, 3.5 * rows))
    for i in range(n):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(images[i])
        plt.xticks([])
        plt.yticks([])

        true_name = class_names[true_labels[i]]
        pred_name = class_names[pred_labels[i]]
        conf = pred_probs[i][pred_labels[i]]

        color = "green" if true_name == pred_name else "red"
        plt.title(
            f"Pred: {pred_name}\nTrue: {true_name}\nConf: {conf:.2f}",
            color=color,
            fontsize=10
        )

    plt.tight_layout()
    plt.show()

# Part A — Load & Explore CIFAR-10

## 1. Load dataset

Isi bagian `TODO` di bawah ini untuk memuat dataset **CIFAR-10**.

### Petunjuk
- Gunakan `keras.datasets.cifar10.load_data()`
- Kalau `USE_SUBSET=True`, ambil sebagian data saja
- Simpan hasilnya ke variabel:
  - `x_train`
  - `y_train`
  - `x_test`
  - `y_test`

### Yang perlu kamu cek
- shape data train
- shape label train
- shape data test
- shape label test

In [ ]:

# ============================================
# 4. TODO — Load CIFAR-10
# ============================================

# TODO:
# 1) Load CIFAR-10
# 2) Terapkan subset bila USE_SUBSET=True
# 3) Simpan ke x_train, y_train, x_test, y_test

x_train, y_train, x_test, y_test = None, None, None, None

# ===== TULIS KODE DI SINI =====



# ===== BATAS TODO =====

if x_train is None or y_train is None or x_test is None or y_test is None:
    raise ValueError("Isi bagian TODO terlebih dahulu.")

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape :", x_test.shape)
print("y_test shape :", y_test.shape)

In [ ]:

# Quick check
assert x_train.ndim == 4 and x_train.shape[1:] == (32, 32, 3), "Shape x_train harus (N, 32, 32, 3)"
assert y_train.ndim == 2 and y_train.shape[1] == 1, "Shape y_train harus (N, 1)"
assert len(np.unique(y_train)) == 10, "Label train harus berisi 10 kelas"
print("✅ Dataset berhasil dimuat dengan format yang benar.")

## 2. Eksplorasi awal dataset

Sekarang tampilkan:
- beberapa gambar sampel dari CIFAR-10,
- nama kelas tiap gambar,
- distribusi label pada data train.

### Tantangan kecil
Coba amati:
- kelas mana yang paling sering muncul?
- apakah distribusinya seimbang?

In [ ]:

# ============================================
# 5. TODO — Visualisasi sample images
# ============================================

# TODO:
# Tampilkan 16 gambar pertama dari x_train
# Tampilkan nama kelasnya menggunakan CLASS_NAMES
# Gunakan figsize yang cukup besar agar rapi

# ===== TULIS KODE DI SINI =====



# ===== BATAS TODO =====

In [ ]:

# Distribusi label
show_class_distribution(y_train, title="Distribusi Label pada Training Set")

# Part B — Preprocessing

Sebelum masuk ke training, data perlu disiapkan.

## Kenapa preprocessing penting?

| Langkah | Tujuan |
|---|---|
| Normalisasi pixel | Membuat training lebih stabil |
| Flatten untuk MLP | Karena MLP menerima input vektor 1D |
| Menjaga bentuk 32×32×3 untuk CNN | Karena CNN butuh struktur spasial gambar |
| Validation split | Supaya kita bisa memantau generalisasi model |

---

## Tugasmu
Isi bagian TODO di bawah untuk:
1. menormalisasi pixel ke rentang **0–1**,
2. membuat versi data untuk **MLP**,
3. membuat versi data untuk **CNN**,
4. membagi training menjadi **train** dan **validation**.

In [ ]:

# ============================================
# 6. TODO — Preprocessing
# ============================================

# Target variabel yang harus tersedia setelah TODO:
# x_train_norm, x_test_norm
# x_train_mlp, x_val_mlp, x_test_mlp
# x_train_cnn, x_val_cnn, x_test_cnn
# y_train_split, y_val_split, y_test_ready

x_train_norm, x_test_norm = None, None
x_train_mlp, x_val_mlp, x_test_mlp = None, None, None
x_train_cnn, x_val_cnn, x_test_cnn = None, None, None
y_train_split, y_val_split, y_test_ready = None, None, None

# ===== TULIS KODE DI SINI =====
# Petunjuk:
# - normalisasi dengan membagi 255.0
# - hitung jumlah data validasi dari VAL_RATIO
# - pakai slicing sederhana
# - untuk MLP: reshape menjadi (N, 32*32*3)
# - untuk CNN: pertahankan bentuk (N, 32, 32, 3)



# ===== BATAS TODO =====

required_vars = [
    x_train_norm, x_test_norm,
    x_train_mlp, x_val_mlp, x_test_mlp,
    x_train_cnn, x_val_cnn, x_test_cnn,
    y_train_split, y_val_split, y_test_ready
]

if any(v is None for v in required_vars):
    raise ValueError("Masih ada variabel preprocessing yang belum diisi.")

print("x_train_mlp:", x_train_mlp.shape)
print("x_val_mlp  :", x_val_mlp.shape)
print("x_test_mlp :", x_test_mlp.shape)
print("x_train_cnn:", x_train_cnn.shape)
print("x_val_cnn  :", x_val_cnn.shape)
print("x_test_cnn :", x_test_cnn.shape)
print("y_train    :", y_train_split.shape)
print("y_val      :", y_val_split.shape)
print("y_test     :", y_test_ready.shape)

In [ ]:

# Quick checks
assert np.max(x_train_norm) <= 1.0 and np.min(x_train_norm) >= 0.0, "Normalisasi belum benar."
assert x_train_mlp.ndim == 2 and x_train_mlp.shape[1] == 32*32*3, "MLP harus pakai input flatten."
assert x_train_cnn.ndim == 4 and x_train_cnn.shape[1:] == (32, 32, 3), "CNN harus pakai input 4D."
print("✅ Preprocessing siap digunakan.")

# Part C — Baseline Model: MLP

MLP akan menjadi **baseline**.  
Model ini tetap bisa dipakai untuk klasifikasi gambar, tetapi ia **tidak mempertahankan struktur spasial** gambar secara eksplisit.

## Tugasmu
Isi TODO untuk:
- membuat arsitektur MLP,
- melakukan compile,
- menjalankan training.

### Saran arsitektur awal
Kamu boleh mulai dari:
- Dense 512
- Dropout
- Dense 256
- Output 10 kelas

> Kamu boleh memodifikasi jumlah neuron, aktivasi, atau dropout jika ingin bereksperimen.

In [ ]:

# ============================================
# 7. TODO — Build MLP
# ============================================

mlp_model = None

# ===== TULIS KODE DI SINI =====
# Petunjuk:
# - gunakan tf.keras.Sequential
# - input shape = 32*32*3
# - output layer harus memiliki 10 neuron
# - gunakan softmax di output



# ===== BATAS TODO =====

if mlp_model is None:
    raise ValueError("MLP model belum dibuat.")

mlp_model.summary()

In [ ]:

# ============================================
# 8. TODO — Compile & Train MLP
# ============================================

history_mlp = None

# ===== TULIS KODE DI SINI =====
# Petunjuk:
# - optimizer: boleh mulai dari Adam
# - loss: sparse_categorical_crossentropy
# - metric: accuracy
# - train menggunakan x_train_mlp dan y_train_split
# - validation menggunakan x_val_mlp dan y_val_split



# ===== BATAS TODO =====

if history_mlp is None:
    raise ValueError("Training MLP belum dijalankan.")

In [ ]:

# Evaluasi awal MLP
mlp_test_loss, mlp_test_acc = mlp_model.evaluate(x_test_mlp, y_test_ready, verbose=0)
print(f"MLP Test Accuracy: {mlp_test_acc:.4f}")
print(f"MLP Test Loss    : {mlp_test_loss:.4f}")

plot_history(history_mlp, title_prefix="MLP")

# Part D — Model untuk Computer Vision: CNN

Sekarang kita masuk ke arsitektur yang lebih cocok untuk citra, yaitu **Convolutional Neural Network (CNN)**.

## Kenapa CNN biasanya lebih cocok?
Karena CNN memproses gambar dengan mempertahankan:
- **lokasi relatif** piksel,
- **pola lokal** seperti edge, texture, shape,
- struktur spasial yang hilang saat gambar langsung di-flatten.

## Tugasmu
Isi TODO untuk:
- membuat CNN sederhana,
- compile model,
- train model.

### Saran blok awal
- Conv2D + ReLU
- MaxPooling2D
- Conv2D + ReLU
- MaxPooling2D
- Flatten
- Dense
- Output softmax

In [ ]:

# ============================================
# 9. TODO — Build CNN
# ============================================

cnn_model = None

# ===== TULIS KODE DI SINI =====
# Petunjuk:
# - input shape = (32, 32, 3)
# - buat minimal 2 convolution block
# - output layer = 10 kelas dengan softmax



# ===== BATAS TODO =====

if cnn_model is None:
    raise ValueError("CNN model belum dibuat.")

cnn_model.summary()

In [ ]:

# ============================================
# 10. TODO — Compile & Train CNN
# ============================================

history_cnn = None

# ===== TULIS KODE DI SINI =====
# Petunjuk:
# - optimizer: Adam
# - loss: sparse_categorical_crossentropy
# - metric: accuracy
# - train dengan x_train_cnn dan y_train_split
# - validation dengan x_val_cnn dan y_val_split



# ===== BATAS TODO =====

if history_cnn is None:
    raise ValueError("Training CNN belum dijalankan.")

In [ ]:

# Evaluasi awal CNN
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_cnn, y_test_ready, verbose=0)
print(f"CNN Test Accuracy: {cnn_test_acc:.4f}")
print(f"CNN Test Loss    : {cnn_test_loss:.4f}")

plot_history(history_cnn, title_prefix="CNN")

# Part E — Bandingkan MLP vs CNN

Bagian ini penting karena inti challenge adalah **membandingkan dua pendekatan** untuk tugas klasifikasi citra yang sama.

## Pertanyaan utama
1. Model mana yang memiliki **accuracy** lebih tinggi?
2. Model mana yang memiliki **gap train-val** lebih kecil?
3. Dari sisi konsep, kenapa hasilnya bisa berbeda?

Isi bagian TODO untuk membuat tabel perbandingan sederhana.

In [ ]:

# ============================================
# 11. TODO — Perbandingan MLP vs CNN
# ============================================

comparison_df = None

# ===== TULIS KODE DI SINI =====
# Petunjuk:
# - buat DataFrame dengan kolom:
#   model, test_loss, test_accuracy, total_params
# - gunakan hasil evaluasi MLP dan CNN
# - total_params bisa diambil dari model.count_params()



# ===== BATAS TODO =====

if comparison_df is None:
    raise ValueError("Tabel perbandingan belum dibuat.")

display(comparison_df.sort_values("test_accuracy", ascending=False))

In [ ]:

# Visualisasi perbandingan akurasi
plt.figure(figsize=(6, 4))
plt.bar(comparison_df["model"], comparison_df["test_accuracy"])
plt.title("Perbandingan Test Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.show()

## Pilih model terbaik

Sekarang pilih model terbaik berdasarkan hasil evaluasi.  
Isi TODO berikut dengan:
- `best_model_name`
- `best_model`
- `x_test_best`

> Biasanya yang dipilih adalah model dengan **test accuracy** paling tinggi, tapi kamu juga boleh mempertimbangkan stabilitas training.

In [ ]:

# ============================================
# 12. TODO — Pilih model terbaik
# ============================================

best_model_name = None
best_model = None
x_test_best = None

# ===== TULIS KODE DI SINI =====



# ===== BATAS TODO =====

if best_model is None or x_test_best is None:
    raise ValueError("Model terbaik belum dipilih.")

print("Best model:", best_model_name)

# Part F — Evaluasi lebih detail

Selain accuracy, kita juga ingin melihat:
- kelas mana yang sering salah diprediksi,
- confusion matrix,
- contoh prediksi benar dan salah.

Jalankan bagian ini setelah model terbaik dipilih.

In [ ]:

# Prediksi pada test set menggunakan model terbaik
y_pred_probs = best_model.predict(x_test_best, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test_ready.reshape(-1)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:

# Confusion matrix
plot_conf_matrix(y_true, y_pred, CLASS_NAMES, title=f"Confusion Matrix - {best_model_name}")

In [ ]:

# Tampilkan sample predictions
show_prediction_grid(x_test, y_test_ready, y_pred_probs, CLASS_NAMES, n=12)

# Part G — Inferensi Model

Bagian ini menunjukkan bagaimana model digunakan untuk **memprediksi gambar baru**.

Untuk challenge ini, kita akan melakukan inferensi pada beberapa sampel dari **test set**.
Kamu boleh mengembangkan bagian ini untuk menguji gambar lain juga.

## Tugasmu
Isi TODO berikut untuk:
- mengambil beberapa indeks acak dari test set,
- melakukan inferensi,
- menampilkan hasil prediksi.

In [ ]:

# ============================================
# 13. TODO — Inferensi pada sample acak
# ============================================

# ===== TULIS KODE DI SINI =====
# Petunjuk:
# - pilih 8 indeks acak dari test set
# - ambil data gambar dan label aslinya
# - lakukan prediksi dengan best_model
# - visualisasikan hasilnya



# ===== BATAS TODO =====

## Bonus opsional — Export model terbaik ke TensorFlow Lite

Kalau kamu selesai lebih cepat, coba kerjakan bonus ini.

### Ide bonus
- konversi `best_model` ke format `.tflite`,
- simpan file model,
- kalau mau, coba jalankan inferensi TFLite juga.

> Bagian ini **opsional**, tapi bagus untuk memahami langkah deployment model ringan.

In [ ]:

# ============================================
# 14. OPTIONAL BONUS — Export ke TFLite
# ============================================

# TODO OPSIONAL:
# 1) Buat converter dari best_model
# 2) Convert ke format TFLite
# 3) Simpan ke file, misalnya "best_cifar10_model.tflite"

# Contoh alur umum:
# converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
# tflite_model = converter.convert()
# with open("best_cifar10_model.tflite", "wb") as f:
#     f.write(tflite_model)

print("Bonus opsional: isi sendiri jika ingin mencoba export ke TFLite.")

# Refleksi akhir

Isi jawaban singkatmu di bawah ini.

## 1. Model mana yang performanya lebih baik?
*Tulis jawabanmu di sini.*

## 2. Menurutmu kenapa CNN biasanya lebih unggul daripada MLP untuk data gambar?
*Tulis jawabanmu di sini.*

## 3. Bagian mana yang paling menantang saat mengerjakan challenge ini?
*Tulis jawabanmu di sini.*

## 4. Kalau diberi waktu lebih, apa yang ingin kamu improve?
Contoh:
- menambah data augmentation,
- tuning arsitektur,
- menambah callback,
- mencoba batch size / learning rate lain,
- export TFLite dan inferensi dari gambar upload.

*Tulis jawabanmu di sini.*

# Checklist sebelum submit / presentasi

Pastikan semua ini sudah ada:

- [ ] dataset CIFAR-10 berhasil dimuat
- [ ] visualisasi sampel gambar muncul
- [ ] preprocessing berhasil untuk MLP dan CNN
- [ ] MLP berhasil di-train
- [ ] CNN berhasil di-train
- [ ] tabel perbandingan MLP vs CNN ada
- [ ] confusion matrix model terbaik ada
- [ ] inferensi model berhasil dijalankan
- [ ] refleksi akhir sudah diisi

Semangat! 🚀